In [ ]:
# Importando Bibliotecas:
import pandas as pd
from pathlib import Path

pasta = Path(".")
padrao = "RONDA_*_2026.CSV"
csvs = sorted(pasta.glob(padrao)) or sorted([*pasta.glob("*.CSV"), *pasta.glob("*.csv")])

if not csvs:
    raise FileNotFoundError(f"Nenhum CSV encontrado em {pasta.resolve()}")

saida_parquet = Path("../dados.parquet")
saida_parquet.parent.mkdir(parents=True, exist_ok=True)

# remove coluna de índice "vazada" do CSV (ex.: Unnamed: 0 / index)
INDEX_COLS = {"index", "unnamed: 0", ""}

def limpar_colunas(cols):
    cols = [str(c).strip() for c in cols]
    return [c for c in cols if c.strip().lower() not in INDEX_COLS]

colunas_ref = None
total = 0
df_final = None

for p in csvs:
    header = pd.read_csv(
        p,
        nrows=0,
        encoding="cp1252",
        sep=";",
        decimal=",",
        encoding_errors="strict"
    )

    cols = limpar_colunas(header.columns)

    if colunas_ref is None:
        colunas_ref = list(cols)
    elif list(cols) != colunas_ref:
        raise ValueError(
            f"Colunas diferentes em: {p}\n"
            f"Esperado: {colunas_ref}\n"
            f"Achado: {list(cols)}"
        )

    chunks = []
    for chunk in pd.read_csv(
        p,
        chunksize=200_000,
        low_memory=False,
        encoding="cp1252",
        sep=",",
        encoding_errors="strict"
    ):
        # limpa nomes e remove colunas de índice se existirem
        chunk.columns = [str(c).strip() for c in chunk.columns]
        drop_cols = [c for c in chunk.columns if c.strip().lower() in INDEX_COLS]
        if drop_cols:
            chunk = chunk.drop(columns=drop_cols)

        chunks.append(chunk)
        total += len(chunk)

    df_arquivo = pd.concat(chunks, ignore_index=True)
    df_final = df_arquivo if df_final is None else pd.concat([df_final, df_arquivo], ignore_index=True)

# garante que o Parquet não salva o índice do pandas
df_final.to_parquet(saida_parquet, index=False)

print(f"✅ OK: {total:,} linhas -> {saida_parquet}")